# AI Red Teaming Framework
### Project 3 of 4 -- AI Governance Portfolio
**Author:** Abhinit Sen | [github.com/abh1nitsen](https://github.com/abh1nitsen) | [LinkedIn](https://www.linkedin.com/in/abhinit-sen-63443015/)

---

This notebook red-teams a Groq-hosted language model across six governance failure categories relevant to regulated Indian industry deployments.

**What you will see:**
- Live probe-by-probe output as each attack runs
- A final robustness score and verdict (ROBUST / NEEDS REVIEW / HIGH RISK)
- An HTML report rendered inline in this notebook

**Requirements:** A free Groq API key added to Colab Secrets (key icon in the left sidebar).

**Runtime:** ~5 minutes for 5 probes per category (30 probes total).

In [ ]:
# Cell 2: Setup -- Upload zip, clone repo, install dependencies
# ---------------------------------------------------------------
# HOW TO USE:
#   Option A (recommended): Upload the project zip to the Colab sidebar,
#   then run this cell. It will unzip automatically.
#
#   Option B: Clone directly from GitHub (requires the repo to be public).
#
# This cell is idempotent -- safe to run multiple times.

import os, zipfile, glob, subprocess, sys

PROJECT_DIR = "/content/ai-red-teaming-framework"

# -- Try to unzip an uploaded archive first --
zips = glob.glob("/content/*.zip")
if zips and not os.path.exists(PROJECT_DIR):
    print(f"Found zip: {zips[0]}")
    with zipfile.ZipFile(zips[0], "r") as z:
        z.extractall("/content/")
    # Handle cases where the zip extracts to a differently named folder
    extracted = [d for d in glob.glob("/content/*/") if "red-team" in d.lower()]
    if extracted:
        os.rename(extracted[0].rstrip("/"), PROJECT_DIR)
    print(f"Unzipped to {PROJECT_DIR}")

# -- Fallback: clone from GitHub --
elif not os.path.exists(PROJECT_DIR):
    subprocess.run([
        "git", "clone",
        "https://github.com/abh1nitsen/ai-red-teaming-framework",
        PROJECT_DIR
    ], check=True)
    print("Cloned from GitHub.")
else:
    print(f"Project directory already exists: {PROJECT_DIR}")

# -- Add to Python path --
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

# -- Install dependencies --
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "groq>=0.28.0", "colorama>=0.4.6", "PyYAML>=6.0"],
               check=True)
print("Dependencies installed.")

In [ ]:
# Cell 3: Load API key and configuration
# ---------------------------------------------------------------

import os, yaml
from google.colab import userdata

# Load GROQ_API_KEY from Colab Secrets
try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("Empty key")
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("[OK] GROQ_API_KEY loaded from Colab Secrets.")
except Exception as e:
    print(f"[FAIL] Could not load GROQ_API_KEY: {e}")
    print()
    print("To add it:")
    print("  1. Click the key icon in the left sidebar")
    print("  2. Add a secret named GROQ_API_KEY")
    print("  3. Paste your Groq API key (free at console.groq.com/keys)")
    print("  4. Re-run this cell")
    raise

# Load config
with open("config/config.yaml") as f:
    config = yaml.safe_load(f)

print(f"[OK] Config loaded.")
print(f"     Probes per category : {list(config['attack_categories'].values())[0].get('probes_per_category', 5)}")
print(f"     Categories enabled  : {sum(1 for v in config['attack_categories'].values() if v.get('enabled', True))}")
print()
print("Config is ready. Proceed to Cell 4 to run the assessment.")

In [ ]:
# Cell 4: Run the red team assessment
# ---------------------------------------------------------------
# Live output prints here as each probe runs.
# The assessment sends adversarial prompts to the target model and
# uses a separate judge model to score each response.
#
# To change the number of probes per category (faster = fewer probes):
# for v in config["attack_categories"].values():
#     v["probes_per_category"] = 3   # change 3 to any number 1-20

from src.runner.red_team_runner import RedTeamRunner
from src.reporting.report_generator import ReportGenerator

runner = RedTeamRunner(config=config, project_root="/content/ai-red-teaming-framework")
result = runner.run(verbose=True)

gen = ReportGenerator(config=config)
paths = gen.generate(result)

print()
print(f"Reports saved:")
if "json" in paths: print(f"  JSON : {paths['json']}")
if "html" in paths: print(f"  HTML : {paths['html']}")

In [ ]:
# Cell 5: View the HTML report inline in this notebook
# ---------------------------------------------------------------

from IPython.display import IFrame, display

if "html" in paths:
    print(f"Displaying: {paths['html']}")
    display(IFrame(src=paths["html"], width="100%", height="900px"))
else:
    print("HTML report was not generated. Check config['reporting']['generate_html'].")

In [ ]:
# Cell 6: Inspect the JSON report (machine-readable, Project 4 compatible)
# ---------------------------------------------------------------

import json

if "json" in paths:
    with open(paths["json"]) as f:
        data = json.load(f)

    print(f"Report ID       : {data['report_id']}")
    print(f"Target model    : {data['target_model']}")
    print(f"Judge model     : {data['judge_model']}")
    print(f"Robustness score: {data['overall']['robustness_score']}")
    print(f"Verdict         : {data['overall']['verdict']}")
    print(f"Total probes    : {data['overall']['total_probes']}")
    print(f"Attacks succeeded: {data['overall']['successful_attacks']}")
    print()
    print("Per-category results:")
    for cat, cat_data in data["categories"].items():
        risk = cat_data.get("risk_rating", "?")
        rate = round(cat_data.get("attack_success_rate", 0) * 100, 1)
        print(f"  {cat:<30} {rate:>5}%  [{risk} RISK]")

In [ ]:
# Cell 7: Deep-dive -- run a single category with detailed output
# ---------------------------------------------------------------
# This is useful for investigating a specific governance failure after
# seeing the full assessment results.
#
# Change CATEGORY to any of:
#   regulatory_boundary | pii_extraction | jailbreak
#   bias_elicitation | hallucination_induction | rag_attack

CATEGORY = "regulatory_boundary"  # <-- Change this

import yaml
with open("/content/ai-red-teaming-framework/config/config.yaml") as f:
    deep_config = yaml.safe_load(f)

# Enable only the selected category
for k, v in deep_config["attack_categories"].items():
    v["enabled"] = (k == CATEGORY)
    v["probes_per_category"] = 10  # Run all available probes

deep_runner = RedTeamRunner(
    config=deep_config,
    project_root="/content/ai-red-teaming-framework"
)
deep_result = deep_runner.run(verbose=True)

In [ ]:
# Cell 8: Run the test suite (offline, no API key required)
# ---------------------------------------------------------------
# All 36 tests use mocked API calls.
# This confirms the framework is correctly installed.

import subprocess
result_proc = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True,
    cwd="/content/ai-red-teaming-framework"
)
print(result_proc.stdout)
if result_proc.returncode != 0:
    print("[FAIL] Some tests failed.")
    print(result_proc.stderr)
else:
    print("[OK] All tests passed.")

## Cell 10 -- Save Project to Google Drive
---------------------------------------------------------------
Copies the project from the Colab session (which is wiped when the
runtime disconnects) to a persistent folder on your Google Drive.

Safe to re-run: files that already match on Drive are skipped, only
new or changed files are copied. Reports, __pycache__, and .git are
excluded (reports regenerate each run; .git is handled by Cell 11).

In [ ]:
# Cell 10: Save Project to Google Drive
# ---------------------------------------------------------------
import os, shutil, filecmp

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/ai-portfolio/project-03-ai-red-teaming"
SOURCE_PATH = "/content/ai-red-teaming-framework"

os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)

SKIP_DIRS = {".git", "__pycache__", ".pytest_cache", "htmlcov"}
SKIP_EXTS = {".pyc", ".pyo"}

copied, skipped, failed = [], [], []

for root, dirs, files in os.walk(SOURCE_PATH):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
    # Skip generated reports (they regenerate on each run)
    rel_root = os.path.relpath(root, SOURCE_PATH)
    if rel_root.startswith("reports") and rel_root != "reports":
        continue

    for fname in files:
        ext = os.path.splitext(fname)[1]
        if ext in SKIP_EXTS:
            continue
        if rel_root == "reports" and fname != ".gitkeep":
            continue

        src_file = os.path.join(root, fname)
        rel_path = os.path.relpath(src_file, SOURCE_PATH)
        dst_file = os.path.join(DRIVE_PROJECT_PATH, rel_path)

        try:
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            if os.path.exists(dst_file) and filecmp.cmp(src_file, dst_file, shallow=False):
                skipped.append(rel_path)
                continue
            shutil.copy2(src_file, dst_file)
            copied.append(rel_path)
        except Exception as e:
            failed.append((rel_path, str(e)))

print(f"[OK] Copied  : {len(copied)} files")
print(f"[OK] Skipped : {len(skipped)} files (identical)")
if failed:
    print(f"[FAIL] Failed : {len(failed)} files")
    for p, e in failed:
        print(f"       {p}: {e}")

print(f"\nProject saved to:")
print(f"  {DRIVE_PROJECT_PATH}")
print(f"\nThis will persist after the Colab runtime disconnects.")


## Cell 11 -- Push to GitHub (fail-safe)
---------------------------------------------------------------
Before running:
1. Create a GitHub Personal Access Token (fine-grained) with
   `Contents: Read & Write` permission, scoped to this repo
   (or "all repositories" if the repo doesn't exist yet)
2. Have the token ready -- it is entered at runtime and hidden,
   never stored in the notebook or on disk

What this cell does, in order:
1. Authenticates with the token and confirms the GitHub username
2. Checks whether the target repository exists
3. If it does not exist, creates it (public, no README/gitignore)
4. If it does exist, uses it as-is -- nothing is overwritten blindly
5. Initialises git locally if needed, using `symbolic-ref` so the
   default branch is `main` regardless of the installed git version
6. Commits only if there are actual changes (skips a no-op commit)
7. Pushes using `HEAD:main` explicitly, with a `--force` fallback
   only if the initial push is rejected (e.g. first push to a repo
   GitHub auto-initialised)
8. Removes the PAT from the git remote URL afterwards so it is never
   left sitting in `.git/config`

> Expected: `[OK] Pushed to GitHub.` followed by the repo URL.

In [ ]:
# Cell 11: Push to GitHub (fail-safe: checks repo, creates if absent)
# ---------------------------------------------------------------
import subprocess, getpass, requests, os

GITHUB_USERNAME = "abh1nitsen"
REPO_NAME       = "ai-red-teaming-framework"
COMMIT_MSG      = "update: AI Red Teaming Framework"

PROJECT_PATH = "/content/ai-red-teaming-framework"

# -- PAT entered at runtime, never stored -------------------------
print("Enter your GitHub Personal Access Token.")
print("(Fine-grained PAT, Contents: Read & Write, for this repo)")
PAT = getpass.getpass("PAT (hidden): ")
if not PAT.strip():
    raise ValueError("No PAT entered. Aborting.")

HEADERS = {
    "Authorization": f"token {PAT}",
    "Accept": "application/vnd.github.v3+json",
}

# -- Step 1: Authenticate ------------------------------------------
print("\nStep 1/4  Authenticating with GitHub...")
r = requests.get("https://api.github.com/user", headers=HEADERS)
if r.status_code != 200:
    raise ValueError(f"Auth failed ({r.status_code}): {r.json().get('message', r.text)}")
AUTH_USER = r.json()["login"]
print(f"  [OK] Authenticated as: {AUTH_USER}")
if AUTH_USER.lower() != GITHUB_USERNAME.lower():
    print(f"  [WARNING] Token belongs to '{AUTH_USER}', not '{GITHUB_USERNAME}'.")
    print(f"            Using '{AUTH_USER}' for the repo owner.")
    GITHUB_USERNAME = AUTH_USER

# -- Step 2: Check if repo exists, create only if it does not -----
print(f"\nStep 2/4  Checking repo {GITHUB_USERNAME}/{REPO_NAME}...")
repo_url = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"
r = requests.get(repo_url, headers=HEADERS)

if r.status_code == 200:
    html_url = r.json()["html_url"]
    print(f"  [OK] Repo already exists: {html_url}")
elif r.status_code == 404:
    print(f"  Repo not found -- creating {REPO_NAME}...")
    payload = {
        "name": REPO_NAME,
        "description": "AI Red Teaming Framework -- AI Governance Portfolio Project 3",
        "private": False,
        "auto_init": False,   # empty repo: no README, no .gitignore, no conflicts
    }
    r2 = requests.post("https://api.github.com/user/repos", headers=HEADERS, json=payload)
    if r2.status_code == 201:
        html_url = r2.json()["html_url"]
        print(f"  [OK] Repo created: {html_url}")
    else:
        raise ValueError(f"Repo creation failed ({r2.status_code}): {r2.json()}")
else:
    raise ValueError(
        f"Unexpected GitHub response ({r.status_code}): {r.json()}. "
        "Check your PAT has Contents: Read & Write permission."
    )

# -- Step 3: Configure git locally ----------------------------------
print("\nStep 3/4  Configuring git...")

def run(cmd):
    return subprocess.run(
        cmd, shell=True, capture_output=True, text=True, cwd=PROJECT_PATH
    )

run(f'git config --global user.email "{GITHUB_USERNAME}@users.noreply.github.com"')
run(f'git config --global user.name "{GITHUB_USERNAME}"')

if not os.path.exists(os.path.join(PROJECT_PATH, ".git")):
    r_init = run("git init")
    # symbolic-ref is used instead of checkout -b: works across all git
    # versions and avoids "already on branch" errors on re-runs.
    run("git symbolic-ref HEAD refs/heads/main")
    print(f"  [OK] Initialised local git repo.")
    if r_init.returncode != 0:
        print(f"       [WARNING] git init stderr: {r_init.stderr.strip()}")
else:
    print("  [OK] Git repo already initialised.")

# -- Step 4: Commit (only if there are changes) and push ------------
print("\nStep 4/4  Committing and pushing...")

run("git add -A")
commit = run(f'git commit -m "{COMMIT_MSG}"')

if "nothing to commit" in commit.stdout:
    print("  [OK] Nothing new to commit -- GitHub is already up to date.")
else:
    first_line = [l for l in commit.stdout.splitlines() if l.strip()]
    print(f"  [OK] Committed: {first_line[0] if first_line else 'done'}")

    push_url = f"https://{PAT}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
    push = subprocess.run(
        ["git", "push", push_url, "HEAD:main"],
        capture_output=True, text=True, cwd=PROJECT_PATH
    )
    if push.returncode != 0:
        # Common on first push if GitHub auto-initialised something,
        # or if history diverged. Force push is safe here because this
        # repo is solely managed by this notebook.
        print(f"  [WARNING] Initial push rejected, retrying with --force...")
        push = subprocess.run(
            ["git", "push", push_url, "HEAD:main", "--force"],
            capture_output=True, text=True, cwd=PROJECT_PATH
        )

    if push.returncode == 0:
        print(f"  [OK] Pushed to GitHub.")
    else:
        print(f"  [FAIL] Push failed:")
        print(f"         {push.stderr.strip()}")

# Set a clean remote URL (without the PAT) so it never sits in .git/config
run("git remote remove origin 2>/dev/null || true")
run(f"git remote add origin https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git")

print(f"\n{'='*55}")
print(f"  {html_url}")
print(f"  CI runs automatically -- check the Actions tab.")
print(f"{'='*55}")
